In [176]:
import importlib
import numpy as np
import torch
import copy
import random
#import matplotlib.pyplot as plt
from mdl_utils import make_graphs, tied_featurize, parse_PDB, create_labels
from mdl_utils import StructureDataset, StructureDatasetPDB, EncoderProteinMPNN, DecoderProteinMPNN, run_dssp
import pandas as pd
#importlib.reload(mdl_utils)
#import Bio.PDB
from Bio.PDB import PDBParser, DSSP, PPBuilder
import pandas as pd
#importlib.reload(Bio.PDB)

In [177]:
seed = 0
path_to_node_model_weights = "../training/exp_020/model_weights/node_log17_KI"
model_name = 'epoch100'
out_folder = "../evaluation/outputs/training_test_output"
num_seq_per_target = 1
batch_size = 1
sampling_temp = "0.1"
#pdb_path = "/WAVE/bio/MD/senior_design_neelm/scfv_native/scfv_native.pdb"
pdb_path = "/WAVE/bio/MD/senior_design_neelm/scfv_native/min/scfv_native.pdb"
max_length = 200000
backbone_noise = 0.00
show_graphs = False
SAE_level = "node"

In [178]:
def run_dssp(pdb_path):
    main_dict = {}
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", str(pdb_path))
    model = structure[0]
    dssp = DSSP(model, str(pdb_path), dssp='/WAVE/bio/anaconda3/envs/pydssp/bin/mkdssp')
    
    for key in dssp.keys():
        try:
            idx = dssp[key][0] - 1
            aa, ss, asa, phi, psi = (
                dssp[key][1], dssp[key][2], dssp[key][3],
                dssp[key][4], dssp[key][5]
            )
            main_dict[idx] = round(asa, 3)
        except (TypeError, KeyError):
            continue
    df = pd.DataFrame(main_dict, index=['ASA']).T
    print(df)
    buried = df.index[df['ASA'] < 0.2].to_list()
    return buried

buried = run_dssp(pdb_path)
print(len(buried))

       ASA
0    1.000
1    0.232
2    0.024
3    0.613
4    0.043
..     ...
247  0.000
248  0.324
249  0.049
250  0.423
251  1.000

[252 rows x 1 columns]
104


In [179]:
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)   

hidden_dim = 128
num_layers = 3 

model_folder_path = path_to_node_model_weights
if model_folder_path[-1] != '/':
    model_folder_path = model_folder_path + '/'

checkpoint_path = model_folder_path + f'{model_name}.pt'

BATCH_COPIES = batch_size
alphabet = 'ACDEFGHIKLMNPQRSTVWYX'   
device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")

pdb_dict_list = parse_PDB(pdb_path, ca_only=False)
dataset_valid = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=max_length)

basic_checkpoint_path = "../training/exp_020/v_48_020.pt"
#checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
checkpoint = torch.load(basic_checkpoint_path, map_location=device, weights_only=False)
node_checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

# Infer expansion size of latent space
size = int(node_checkpoint['model_state_dict']['sae_layers.0.WS1.weight'].shape[0] / 128)
encoder_model = EncoderProteinMPNN(num_letters=21,
                    node_features=hidden_dim,
                    edge_features=hidden_dim, 
                    hidden_dim=hidden_dim,
                    expansion=size,
                    num_encoder_layers=num_layers, 
                    num_decoder_layers=num_layers, 
                    augment_eps=backbone_noise, 
                    k_neighbors=checkpoint['num_edges'])
encoder_model.to(device)    

encoder_model.load_state_dict(checkpoint['model_state_dict'], strict=False)


filtered_node_state_dict = {}
for k, v in node_checkpoint['model_state_dict'].items():
    if 'sae_layers' in k:
      k = k[11:]
      filtered_node_state_dict[k] = v
encoder_model.node_sae_layers.load_state_dict(filtered_node_state_dict, strict=True)


edge_checkpoint_path = "../training/exp_020/model_weights/edge_log15_KI/epoch_last.pt"
edge_checkpoint = torch.load(edge_checkpoint_path, map_location=device, weights_only=False)

filtered_state_dict = {}
for k, v in edge_checkpoint['model_state_dict'].items():
    if 'sae_layers' in k:
      k = k[11:]
      filtered_state_dict[k] = v
encoder_model.edge_sae_layers.load_state_dict(filtered_state_dict, strict=True)

encoder_model.eval()

decoder_model = DecoderProteinMPNN(num_letters=21,
                    node_features=hidden_dim,
                    edge_features=hidden_dim, 
                    hidden_dim=hidden_dim,
                    expansion=size,
                    num_encoder_layers=num_layers, 
                    num_decoder_layers=num_layers, 
                    augment_eps=backbone_noise, 
                    k_neighbors=48)
decoder_model.to(device)    
#basic_checkpoint_path = "../training/exp_020/v_48_020.pt"
basic_checkpoint = torch.load(basic_checkpoint_path, map_location=device, weights_only=False)
decoder_model.load_state_dict(basic_checkpoint['model_state_dict'], strict=False)

decoder_model.eval();

In [180]:
with torch.no_grad():
    for ix, protein in enumerate(dataset_valid):
        batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
        X, S, mask,_, chain_M, chain_encoding_all,_,_,_,_, chain_M_pos,_, residue_idx,_,_,_,_,_,_,_ = tied_featurize(batch_clones, device, chain_dict=None)
        if SAE_level == 'node':
            error, res_labels = create_labels(pdb_path, SAE_level)
            if error != True:
                randn_1 = torch.randn(chain_M.shape, device=device)
                h_V, h_E, n_original, n_encoded, n_decoded, E_idx, e_original, e_encoded, e_decoded = encoder_model(X, mask, residue_idx, chain_encoding_all)
            

test2
test2
test2
test2
test2
test2


In [181]:
if show_graphs:
    model_name = '/'.join(path_to_node_model_weights.split('/')[-1:])
    protein_name = pdb_path[-8:-4]
    layer = 2
    graph_info = [model_name, protein_name, layer, 'Node']
    make_graphs(n_original, "Original", 2, graph_info)
    make_graphs(n_encoded, "Encoded", 2, graph_info)
    make_graphs(n_decoded, "Decoded", 2, graph_info)

In [182]:
print(h_V.shape)
print(encoder_model.node_input_act[2].shape)
# log 15 edge feature correlations 0 > 0.70
short_dist = [126, 154, 225] # neigh 3: 0.816, 0.792, 0.859
med_dist = [13, 147, 28]# neigh 22-35, 25-33, 26-33
long_dist = [34, 71, 113, 199, 118] # neigh 30-48, 37-48, 40-48, 40-48, 42-48

# log 15 edge feature correlations 1 > 0.70
med_dist = [143, 120, 188, 179] # neigh 20-34, 29-40, 29-48, 30-48
long_dist = [121, 36, 129, 133] #neigh 31-48, 37-48, 37-48, 39-48

# log 15 edge feature correlations 2 > 0.70
long_dist = [92, 103, 249, 80] # neigh 27-48 (38), 34-48 (40), 35-48 (46), 44-48 (80)

# log 17 node feature correlations:
bcaa = [46, 48, 182, 191, 221]

mask_for_empty = np.asarray((S[0] != 20).cpu())
node_encoded = np.round(encoder_model.node_encoded_act[2].cpu().numpy()[0,:,:], decimals=5)[mask_for_empty] # (N, 256)
edge_encoded = np.round(encoder_model.edge_encoded_act[2].cpu().numpy()[0,:,:,:], decimals=5)[mask_for_empty] # (N, 48, 256)

'''
N = h_V.shape[1]
# Average sparse encodings for all residues for log 17 node model
model = 'log17_node_exp2_100e_2'
avg_encodings_path = '/WAVE/bio/ML/SAE_train/SAEProteinMPNN/evaluation/created_data/encodings/ki_encodings/avg_' + model + '.csv'
avg_encodings = pd.read_csv(avg_encodings_path, header=None).T # (256, ) -> (, 256)
node_encoded = pd.concat([avg_encodings] * N).to_numpy() # (N, 256)

# Average sparse edge encodings for all residues for log 15 edge model
model = 'log15_edge_exp2_100e_2'
avg_edge_encodings_path = '/WAVE/bio/ML/SAE_train/SAEProteinMPNN/evaluation/created_data/encodings/ki_encodings/avg_edge_' + model + '.csv'
avg_edge_encodings = pd.read_csv(avg_edge_encodings_path, header=None).to_numpy() # (48, 256)
edge_encoded = np.repeat(avg_edge_encodings[np.newaxis, :, :], N, axis=0) # (N, 48, 256)
'''
'''
for i in range(node_encoded.shape[0]):
    if ((node_encoded[i, 221] - node_encoded[:, 221].min()) / (node_encoded[:, 221].max() - node_encoded[:, 221].min())) > 0.3: # phobic
        print(i)
        node_encoded[i, 131] =0#*= -1*change # unstructured
        node_encoded[i, 182] =2#*= change #phobic
        edge_encoded[i, 27:48, 92] =0#*= -1*change # long edge
        edge_encoded[i, 34:48, 103] =0#*= -1*change # long edge
        edge_encoded[i, 35:48, 249] =0#*= -1*change # long edge
        edge_encoded[i, 44:48, 80] =0#*= -1*change # long edge
'''
edge_encoded_ = torch.tensor(np.reshape(edge_encoded, (1, -1, 48, 256)))
W2_edge = encoder_model.edge_sae_layers[2].WS2.weight
b2_edge = encoder_model.edge_sae_layers[2].WS2.bias
W1_edge = encoder_model.edge_sae_layers[2].WS1.weight
b1_edge = encoder_model.edge_sae_layers[2].WS1.bias
W1_edge_pinv = torch.linalg.pinv(W1_edge)

edge_encoded_ = edge_encoded_.to(W2_edge.dtype)
edge_encoded_ = edge_encoded_ @ W2_edge.T + b2_edge
#edge_modified = ((edge_modified - b1_edge) @ W1_edge_pinv.T) + b2_edge 
print(edge_encoded_.shape)

node_encoded_ = torch.tensor(np.reshape(node_encoded, (1, -1, 256)))

W2_node = encoder_model.node_sae_layers[2].WS2.weight
b2_node = encoder_model.node_sae_layers[2].WS2.bias
W1_node = encoder_model.node_sae_layers[2].WS1.weight
b1_node = encoder_model.node_sae_layers[2].WS1.bias
W2_node_pinv = torch.linalg.pinv(W1_node)

node_encoded_ = node_encoded_.to(W2_node.dtype)
node_encoded_ = node_encoded_ @ W2_node.T + b2_node
#node_modified = ((node_modified - b1_node) @ W2_node_pinv.T) + b2_node
print(node_encoded_.shape)


torch.Size([1, 252, 128])
torch.Size([1, 252, 128])
torch.Size([1, 252, 48, 128])
torch.Size([1, 252, 128])


In [ ]:

change = 10



In [184]:

for i in buried:
    #print(node_encoded[i, bcaa])
    node_encoded[i, bcaa] += change
    #rint(node_encoded[i, bcaa])

edge_modified = torch.tensor(np.reshape(edge_encoded, (1, -1, 48, 256)))
edge_modified = edge_modified.to(W2_edge.dtype)
edge_modified = edge_modified @ W2_edge.T + b2_edge
#edge_modified = ((edge_modified - b1_edge) @ W1_edge_pinv.T) + b2_edge 


node_modified = torch.tensor(np.reshape(node_encoded, (1, -1, 256)))
node_modified = node_modified.to(W2_node.dtype)
node_modified = node_modified @ W2_node.T + b2_node
#node_modified = ((node_modified - b1_node) @ W2_node_pinv.T) + b2_node

print(buried)
print(h_V.shape)

[2, 4, 6, 13, 19, 21, 23, 25, 29, 33, 34, 35, 36, 37, 38, 43, 44, 46, 47, 48, 51, 55, 58, 62, 64, 71, 73, 74, 75, 78, 82, 83, 84, 86, 87, 88, 89, 90, 91, 94, 97, 98, 99, 100, 101, 102, 104, 105, 107, 135, 137, 143, 151, 153, 155, 158, 160, 163, 165, 166, 167, 168, 169, 170, 171, 177, 179, 180, 181, 182, 183, 184, 192, 195, 198, 199, 201, 203, 208, 209, 210, 211, 212, 214, 217, 221, 223, 225, 226, 227, 228, 229, 230, 231, 234, 235, 237, 238, 240, 241, 242, 245, 247, 249]
torch.Size([1, 252, 128])


In [185]:
torch.manual_seed(0)
with torch.no_grad():
    for ix, protein in enumerate(dataset_valid):
        batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
        randn_1 = torch.randn(chain_M.shape, device=device)
        log_probs = decoder_model(X, S, mask, chain_M*chain_M_pos, h_V, h_E, E_idx, randn_1)

    S_new = torch.argmax(log_probs, dim=2)
    alphabet = 'ACDEFGHIKLMNPQRSTVWYX'
    seq = ''.join([alphabet[c] for c in S_new[0]])
    print(f"Original\nWith SAE\nModified\n{seq}")
    for ix, protein in enumerate(dataset_valid):
        batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
        randn_1 = torch.randn(chain_M.shape, device=device)
        log_probs = decoder_model(X, S, mask, chain_M*chain_M_pos, node_encoded_, edge_encoded_, E_idx, randn_1)

    S_new = torch.argmax(log_probs, dim=2)
    alphabet = 'ACDEFGHIKLMNPQRSTVWYX'
    seq = ''.join([alphabet[c] for c in S_new[0]])
    print(f"{seq}")

    for ix, protein in enumerate(dataset_valid):
        batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
        randn_1 = torch.randn(chain_M.shape, device=device)
        log_probs = decoder_model(X, S, mask, chain_M*chain_M_pos, node_modified, edge_modified, E_idx, randn_1)

    S_new = torch.argmax(log_probs, dim=2)
    alphabet = 'ACDEFGHIKLMNPQRSTVWYX'
    seq = ''.join([alphabet[c] for c in S_new[0]])
    print(f"{seq}")


Original
With SAE
Modified
AEVEMTQSPLSLEARVGDTVVITCTASEDVGTDVSWYQQWPGQPPQLLIYNASTLAPGVSSRFQGSGSGSNYTLTISSLQAEDFATYYCQNVYNPEVRGMQFGQGTLLTLKGASPESASSPSSVVVPSSPVPPKKLTESGGGTVKPGGSVTLSCKFSGFSLSDYDYLSWIRQAPGKGLEWVGHIGQNGVSYLAPDAKGRFTLSRDLSKNTLYLDMNDLQPEDTAVYYCGLSNDSSGLGFDLWGEGTTVTVAA
METVMTQSPSELEANIGDKVTIRCKASQDVGNAVSWYKQLPGEPPELIIYGASVLMKGVPSRYKGSGEGSKYTLTISELQEEDFAVYYCQNTYKPEKNGRQFGEGTNVTEKGDEGSDKKKSVEKKKPKVDKKKETLKESGGGTVEPGGSVTLSCKATGCNLSDYEYNSWYRQRPGKGWRFVGHIDRNGKSYLAEEAKGRWTISVDLENNTMYLNMNNLQEEDTAVYYASLRSEESGLALSVWGQGTKVTVAE
METKVTQSPKSHSANIGETTTLKCKASSDTGTYVSFLHQKPGEPPRLLISGASKLAPGVPSRYKGSGYGTKYTLTISELKADDFAVYECQNTYKPEKNGAQYGQGTKITRKGAKGTGVPSPVGKRKPKVGKKKRTLTETGGGTVKPGGSRTLSGKASGLSLSDLDHTSVRSQRPGGGPESLGHIHRNGKSYLSEEAKGRVTLSRDYENNTAHHTMRDLQEEDTAVYECTRMNEKSGYAQSLTGQGNQVTVEA
